In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import re



In [24]:
name = 'csv/test031_IKKI_A_5CH.csv'
num_CH = 5
CH_first = 0

In [25]:
df = pd.read_csv(name, sep=';', dtype=str)
df["tiempos"]  = df.index
df = df.reset_index(drop=True)
df.columns

Index(['Time', 'SPI (2,3,1,4)', 'tiempos'], dtype='object')

In [26]:
print(df.head(5))

   Time SPI (2,3,1,4)      tiempos
0  0x88           NaN  0,000025745
1  0xbb           NaN  0,000031380
2  0x32           NaN  0,000037655
3  0x7f           NaN  0,000090110
4  0xb4           NaN  0,000095760


In [27]:
df = df.drop(columns = 'SPI (2,3,1,4)')

In [28]:
df.columns = ['MOSI', 'tiempos']

print(df.head(21))


    MOSI      tiempos
0   0x88  0,000025745
1   0xbb  0,000031380
2   0x32  0,000037655
3   0x7f  0,000090110
4   0xb4  0,000095760
5   0x33  0,000102050
6   0x7f  0,000154275
7   0xb5  0,000159880
8   0x34  0,000166130
9   0x88  0,000218800
10  0x2a  0,000224425
11  0x30  0,000230680
12  0xa1  0,000282320
13  0x2d  0,000287965
14  0x31  0,000294255
15  0x89  0,000346700
16  0x41  0,000352325
17  0x32  0,000358580
18  0x7f  0,000410490
19  0xdd  0,000416095
20  0x33  0,000422340


In [29]:
df['MOSI'] = df['MOSI'].apply(lambda x: int(x, 16))

In [30]:
# Verifica cómo quedó
print(df.head(10))


   MOSI      tiempos
0   136  0,000025745
1   187  0,000031380
2    50  0,000037655
3   127  0,000090110
4   180  0,000095760
5    51  0,000102050
6   127  0,000154275
7   181  0,000159880
8    52  0,000166130
9   136  0,000218800


In [31]:
# Crea un diccionario para mapear los valores
mapeo = {
    48+CH_first: CH_first
}

# Comienza después de la última clave
ultima_clave = max(mapeo.keys())
ultimo_valor = max(mapeo.values())

for i in range(1, num_CH):
    nueva_clave = ultima_clave + i
    nuevo_valor = ultimo_valor + i
    mapeo[nueva_clave] = nuevo_valor

print(mapeo)

{48: 0, 49: 1, 50: 2, 51: 3, 52: 4}


In [32]:
# Aplica el mapeo y pon 'S' en el resto
df['type'] = df['MOSI'].astype(int).map(mapeo).fillna('S')
df = df.reset_index(drop=True)
print(df.head(40))


    MOSI      tiempos type
0    136  0,000025745    S
1    187  0,000031380    S
2     50  0,000037655  2.0
3    127  0,000090110    S
4    180  0,000095760    S
5     51  0,000102050  3.0
6    127  0,000154275    S
7    181  0,000159880    S
8     52  0,000166130  4.0
9    136  0,000218800    S
10    42  0,000224425    S
11    48  0,000230680  0.0
12   161  0,000282320    S
13    45  0,000287965    S
14    49  0,000294255  1.0
15   137  0,000346700    S
16    65  0,000352325    S
17    50  0,000358580  2.0
18   127  0,000410490    S
19   221  0,000416095    S
20    51  0,000422340  3.0
21   127  0,000474835    S
22   172  0,000480485    S
23    52  0,000486780  4.0
24   136  0,000539560    S
25   105  0,000545200    S
26    48  0,000551460  0.0
27   160  0,000603220    S
28   249  0,000608865    S
29    49  0,000615150  1.0
30   136  0,000667410    S
31   247  0,000673060    S
32    50  0,000679345  2.0
33   127  0,000731835    S
34   234  0,000737455    S
35    51  0,000743715  3.0
3

In [33]:
# Inicializamos idx como None
idx = None
# Recorremos los índices donde 'type' es distinto de 'S'
for i in df.index[df['type'] != 'S']:
    # Verificamos que haya al menos dos filas siguientes
    if (i + 2) < len(df):
        # Comprobamos que las dos siguientes filas tengan 'S'
        if (df.loc[i + 1, 'type'] == 'S') and (df.loc[i + 2, 'type'] == 'S'):
            idx = i
            break
    else:
        # Si no hay suficientes filas para verificar, no es un punto válido
        continue

# Si encontramos un índice válido, cortamos el dataframe
if idx is not None:
    df = df.loc[idx:].reset_index(drop=True)
else:
    # Si no se encontró un punto válido, el dataframe queda vacío o como prefieras manejarlo
    df = df.iloc[0:0].reset_index(drop=True)


In [34]:
print(df['type'][0])
print(df['type'][1])
print(df['type'][2])
print(df['type'][3])


2.0
S
S
3.0


In [35]:
# Diccionario para almacenar los arrays
resultados = {CH_first: []}
resultados_tiempos = {CH_first: []}



# Obtener el valor máximo actual de clave
ultima_clave = max(resultados.keys())

# Agregar N nuevas claves a ambos diccionarios
for i in range(1, num_CH):
    nueva_clave = ultima_clave + i
    resultados[nueva_clave] = []
    resultados_tiempos[nueva_clave] = []

# Iterar sobre el dataframe
for i, row in df.iterrows():
    tipo = row['type']
    tipo_tiempo = row['type']  
    if tipo in resultados:
        # Tomar las dos siguientes filas si existen
        sub_df = df.iloc[i+1:i+3]['MOSI']
        sub_df_times = df.iloc[i+1:i+3]['tiempos']
        # Guardar como array (puedes ajustar qué columnas guardar)
        resultados[tipo].append(sub_df.to_numpy())
        resultados_tiempos[tipo_tiempo].append(sub_df_times.to_numpy())
        




In [36]:

# Opcional: convertir las listas en arrays grandes (si quieres)
import numpy as np
for k in resultados:
    resultados[k] = np.concatenate(resultados[k])
    resultados_tiempos[k] = np.concatenate(resultados_tiempos[k])

# Ahora resultados[0], resultados[1], ... tienen los arrays deseados

In [37]:
for i in range (num_CH):
    print(len(resultados[i]))


626
626
636
630
634


In [38]:
# Creamos un nuevo diccionario con las filas pares eliminadas
resultados_filtrados = {}

for k, arr in resultados_tiempos.items():
    # Tomar los elementos en posiciones impares: 1, 3, 5, ...
    resultados_tiempos[k] = arr[1::2]


In [39]:
arr0 = resultados[0].flatten()

new_arr0 = []
new_arr1 = []
new_arr2 = []
new_arr3 = []
new_arr4 = []
for i in range(0, len(arr0)-1, 2):
    combined = arr0[i] * 256 + arr0[i+1]
    new_arr0.append(combined)
new_arr0 = np.array(new_arr0)



In [40]:
arr1 = resultados[1].flatten()
for i in range(0, len(arr1)-1, 2):
    combined = arr1[i] * 256 + arr1[i+1]
    new_arr1.append(combined)
new_arr1 = np.array(new_arr1)


In [41]:

arr2 = resultados[2].flatten()
for i in range(0, len(arr2)-1, 2):
    combined = arr2[i] * 256 + arr2[i+1]
    new_arr2.append(combined)
new_arr2 = np.array(new_arr2)

In [42]:

arr3 = resultados[3].flatten()
for i in range(0, len(arr3)-1, 2):
    combined = arr3[i] * 256 + arr3[i+1]
    new_arr3.append(combined)
new_arr3 = np.array(new_arr3)

In [43]:

arr4 = resultados[4].flatten()
for i in range(0, len(arr4)-1, 2):
    combined = arr4[i] * 256 + arr4[i+1]
    new_arr4.append(combined)
new_arr4 = np.array(new_arr4)

In [ ]:

fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(y=new_arr0, mode='lines', name='Señal 0'))
fig.add_trace(go.Scatter(y=new_arr1, mode='lines', name='Señal 1'))
fig.add_trace(go.Scatter(y=new_arr2, mode='lines', name='Señal 2'))
fig.add_trace(go.Scatter(y=new_arr3, mode='lines', name='Señal 3'))
fig.add_trace(go.Scatter(y=new_arr4, mode='lines', name='Señal 4'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()


In [45]:

fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[0], y=new_arr0, mode='lines', name='Señal 0'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()

In [46]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[1], y=new_arr1, mode='lines', name='Señal 1'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()

In [47]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[2], y=new_arr2, mode='lines', name='Señal 2'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()

In [48]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[3], y=new_arr3, mode='lines', name='Señal 3'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()

In [49]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[4], y=new_arr4, mode='lines', name='Señal 4'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()

In [50]:
ceros = np.zeros(100)  
# FFT
X = np.fft.fft(np.concatenate((ceros,(new_arr-32768))))

# Número de muestras
N = len(X)

# Frecuencias asociadas (eje x)
freqs = np.fft.fftfreq(N, 1/f)

# Magnitud (módulo)
magnitud = np.abs(X)

# Para mostrar solo la mitad positiva (frecuencias positivas)
idxs = freqs >= 0

plt.plot(freqs[idxs], magnitud[idxs])
plt.xlabel("Frecuencia (Hz)")
plt.ylabel("Magnitud")
plt.title("Espectro de la señal")
plt.show()

NameError: name 'new_arr' is not defined

In [ ]:

fig = go.Figure()

fig.add_trace(go.Scatter(
    x = freqs[idxs],
    y=magnitud[idxs],
    mode='lines',
    name='Valores concatenados'
))

fig.update_layout(
    title='FFT',
    xaxis_title='Frecuencia',
    yaxis_title='Magnitud',
    hovermode='x unified'
)

fig.show()